# LeNet-1 (1989) Implementation

This notebook is reimplementation of the original Convnet paper from LeCun et.al used for digit recognition in handwritten zip codes [^1]. The neural net was then evolved from this first version over a decade [^2].

## References
[^1]: LeCun, Y. et al. (1989). Backpropagation applied to Handwritten Zip Code Recognition. [Link to PDF](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf)

[^2]: LeCun, Y. et al. (1998) GradientBased Learning Applied to Document Recognition



In [53]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Table of Contents
1. [Data Preparation](#data-preparation)
2. [Model Architecture](#model-architecture)
3. [Forward Pass](#forward-pass)
4. [Training Loop](#training-loop)
5. [Results](#results)

## Data Preparation

The data pre-processing is done in the notebook **pytorch_playground**, which load, resizes to 16x16 image from 28x28 image in MNIST Dataset, and stores the subset of the dataset to npz file in data directory.
* The saved dataset contains 7291 training and 2007 test images of handwritten digits and corresponding labels (0-9)

In [72]:
data_train = np.load("./data/train1989.npz")
data_test = np.load("./data/test1989.npz")

# Access arrays inside
Xtr = torch.from_numpy(data_train["X"])
Ytr = torch.from_numpy(data_train["Y"])

Xte = torch.from_numpy(data_test["X"])
Yte = torch.from_numpy(data_test["Y"])

**Task 4** Check the shape of the training and test data and labels.

In [68]:
print(f"Training data shape: {Xtr.shape}")
print(f"Training labels shape: {Ytr.shape}")
print(f"Test data shape: {Xte.shape}")
print(f"Test labels shape: {Yte.shape}")

Training data shape: torch.Size([7291, 1, 16, 16])
Training labels shape: torch.Size([7291, 10])
Test data shape: torch.Size([2007, 1, 16, 16])
Test labels shape: torch.Size([2007, 10])


## Model Architecture

The notebook uses paper implementation of
* the weight initialization which requires fan_in(number of input connections to a neuron) which prevents vanishing gradients.

* LeNet-1 Network Class
    * image batch: (7291, 20, 20, 1)→ interpreted as NHWC (batch, height, width, channels)
    * weights: (12, 1, 5, 5)→ interpreted as (out_channels, in_channels, filter_height, filter_width)
    * you must tell JAX which dimension format you’re using. Correct usage: NHWC input, OIHW kernel. Output dimension same as bias, NCHW



In [81]:
def init_weights(fan_in, *shape):
    """Weight initialization as described in the paper"""
    weight = (torch.rand(*shape) - 0.5) * 2 * 2.4 / fan_in
    return weight

class Net(nn.Module):
    """1989 LeCun ConvNet per description in the paper"""

    def __init__(self, seed=42):
        super(Net, self).__init__()
        torch.manual_seed(seed)

        macs = 0  # keep track of MACs (multiply accumulates)
        acts = 0  # keep track of number of activations

        # H1 layer parameters
        self.H1w = nn.Parameter(init_weights(5*5*1, 12, 1, 5, 5))
        self.H1b = nn.Parameter(torch.zeros(12, 8, 8))
        assert self.H1w.numel() + self.H1b.numel() == 1068
        macs = 5*5*8*8*12 + 768
        acts += (8*8) * 12
        assert macs == 19968

        # H2 layer parameters
        self.H2w = nn.Parameter(init_weights(5*5*8, 12, 8, 5, 5))
        self.H2b = nn.Parameter(torch.zeros(12, 4, 4))
        assert self.H2w.numel() + self.H2b.numel() == 2592
        macs += (5*5*8) * (4*4) * 12 + 192
        acts += (4*4) * 12
        assert macs == 58560

        # H3 fully connected layer
        self.H3w = nn.Parameter(init_weights(4*4*12, 4*4*12, 30))
        self.H3b = nn.Parameter(torch.zeros(30))
        assert self.H3w.numel() + self.H3b.numel() == 5790
        macs += (4*4*12) * 30 + 30
        acts += 30
        assert macs == 64350

        # Output layer
        self.outw = nn.Parameter(init_weights(30, 30, 10))
        self.outb = nn.Parameter(-torch.ones(10))
        assert self.outw.numel() + self.outb.numel() == 310
        macs += 30 * 10 + 10
        acts += 10
        assert macs == 64660

        self.macs = macs
        self.acts = acts

    def count_params(self):
        """Count total number of parameters"""
        return sum(p.numel() for p in self.parameters())

    def forward(self, x):
        """
        Forward pass - you'll need to implement this based on your network architecture.
        This is a placeholder that shows the structure.
        """
        x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
        # H1 layer
        # Apply convolution with H1w and add H1b
        # Then apply activation and pooling as needed
        x = F.conv2d(x, self.H1w, stride=2) + self.H1b
        x = torch.tanh(x)

        # H2 layer
        # Apply convolution with H2w and add H2b
        # Then apply activation and pooling as needed
        x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
        slice1 = F.conv2d(x[:, 0:8], self.H2w[0:4], stride=2)
        slice2 = F.conv2d(x[:, 4:12], self.H2w[4:8], stride=2)
        slice3 = F.conv2d(torch.cat([x[:, 0:4], x[:, 8:12]], dim=1), self.H2w[8:12], stride=2)
        x = torch.cat((slice1, slice2, slice3), dim=1) + self.H2b
        x = torch.tanh(x)

        # H3 fully connected layer
        x = x.flatten(start_dim=1) # (1, 12*4*4)
        x = x @ self.H3w + self.H3b
        x = torch.tanh(x)

        # x is now shape (1, 30)
        x = x @ self.outw + self.outb
        x = torch.tanh(x)

         # x is finally shape (1, 10)
        return x

        return x

**Task 5** Find out the following features of the model architecture

    * #Parameters: 9760
    * #macs: 64,660
    * #activations:
    * Layer: Conv-Conv-FC-FC

Key concepts

* Padding
* Convolution
* Matrix Multiplication
* Activation

In [82]:
def loss_fn(model, x, y):
    """Compute MSE loss"""
    yhat = model(x)
    return torch.mean((y - yhat) ** 2)


def train_step(model, optimizer, x, y):
    """Single training step"""
    # Zero gradients
    optimizer.zero_grad()

    # Forward pass and compute loss
    yhat = model(x)
    loss = torch.mean((y - yhat) ** 2)

    # Backward pass
    loss.backward()

    # Update parameters
    optimizer.step()

    return loss.item()


def eval_split(model, split, pass_num, X_tr, Y_tr, X_te, Y_te):
    """Evaluate the full train/test set"""
    model.eval()  # Set to evaluation mode

    with torch.no_grad():  # Disable gradient computation
        X, Y = (X_tr, Y_tr) if split == 'train' else (X_te, Y_te)
        Yhat = model(X)
        loss = torch.mean((Y - Yhat)**2)
        err = torch.mean((Y.argmax(dim=1) != Yhat.argmax(dim=1)).float())
        misses = int(err.item() * Y.shape[0])

        print(f"eval: split {split:5s}. loss {loss.item():e}. error {err.item()*100:.2f}%. misses: {misses}")
        model.train()  # Set back to training mode
    return loss.item(), err.item(), misses


In [84]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Parameters
seed = 42
num_epochs = 23


model = Net(seed=seed)
model.train()  # Set back to training mode
optimizer = optim.SGD(model.parameters(), lr=0.03)

for pass_num in range(num_epochs):

    # Perform one epoch of training
    for step_num in range(Xtr.shape[0]):

        # Fetch a single example into a batch of 1
        x, y = Xtr[step_num:step_num+1], Ytr[step_num:step_num+1]

        # Training step
        loss = train_step(model, optimizer, x, y)

    # After each epoch evaluate the train and test error/metrics
    print(f"\nEpoch {pass_num + 1}/{num_epochs}")
    train_loss, train_err, train_misses = eval_split(model, 'train', pass_num, Xtr, Ytr, Xte, Yte)
    test_loss, test_err, test_misses = eval_split(model, 'test', pass_num, Xtr, Ytr, Xte, Yte)


Epoch 1/23
eval: split train. loss 2.672640e-01. error 49.38%. misses: 3599
eval: split test . loss 2.658389e-01. error 49.58%. misses: 994

Epoch 2/23
eval: split train. loss 6.611320e-02. error 9.70%. misses: 707
eval: split test . loss 6.289694e-02. error 8.97%. misses: 179

Epoch 3/23
eval: split train. loss 4.669868e-02. error 6.83%. misses: 498
eval: split test . loss 4.676422e-02. error 6.98%. misses: 139

Epoch 4/23
eval: split train. loss 3.862600e-02. error 5.47%. misses: 398
eval: split test . loss 4.069002e-02. error 6.03%. misses: 120

Epoch 5/23
eval: split train. loss 3.375371e-02. error 4.94%. misses: 360
eval: split test . loss 3.772741e-02. error 5.73%. misses: 115

Epoch 6/23
eval: split train. loss 3.090480e-02. error 4.53%. misses: 330
eval: split test . loss 3.690541e-02. error 5.58%. misses: 112

Epoch 7/23
eval: split train. loss 2.692368e-02. error 3.96%. misses: 288
eval: split test . loss 3.525820e-02. error 5.43%. misses: 109

Epoch 8/23
eval: split train. 